# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets in the dataset with their @id and fields
if not metadata.record_sets:
    print("No record sets were found in the metadata.\n\nObtaining record sets via .record_sets property via mlcroissant's Dataset object...")
    record_sets = list(dataset.record_sets)
else:
    record_sets = metadata.record_sets

print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f'RecordSet @id: {rs.id}")
    print(f'  Name         : {rs.name}')
    print(f'  Description  : {rs.description if hasattr(rs,"description") else ""}')
    # List fields for each record set
    field_ids = []
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            print(f"    Field @id: {f.id}  Name: {f.name}  DataType: {getattr(f, 'data_type', '?')}")
            field_ids.append(f.id)
    else:
        print("  (No fields found)")
    print()
if not record_sets:
    print("No record sets found for extraction. The dataset may be empty or only contain metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set and store as DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for rs in record_sets:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs.id))
        dataframes[rs.id] = df
        print(f'RecordSet @id: {rs.id} - Loaded with shape: {df.shape}')
        print(f'Columns: {df.columns.tolist()}')
    except Exception as e:
        print(f"Failed to load records for record set {rs.id}: {e}")

# Show the first 5 rows for the first record set if it exists
if record_set_ids:
    sample_id = record_set_ids[0]
    if sample_id in dataframes:
        display_cols = dataframes[sample_id].columns.tolist()
        print(f'Columns in @id {sample_id}: {display_cols}')
        dataframes[sample_id].head()
    else:
        print(f"No data found for record set {sample_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose example record set and field for EDA (edit IDs if desired based on overview above)
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    # Try to find a numeric field in columns (case-insensitive match for common protein data fields)
    candidate_fields = ['MW', 'Molecular Weight', 'Abundance', 'Coverage', 'Peptide counts', 'pI']
    numeric_field = None
    for col in df.columns:
        if any(any(k.lower() in col.lower() for k in candidate_fields)):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    # If none found, fall back to first numeric column
    if numeric_field is None:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break

    if numeric_field is None:
        print("No clear numeric field found for EDA.")
    else:
        print(f"Analyzing numeric field: {numeric_field}")

        # Filter, normalize, group
        threshold = df[numeric_field].quantile(0.75)  # use the upper quartile as example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (upper quartile):")
        print(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f'{numeric_field}_normalized'] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

        # Try to group by another categorical field if available
        candidate_groups = ['Sample', 'Group', 'Description', 'Type']
        group_field = None
        for col in df.columns:
            if any(any(k.lower() in col.lower() for k in candidate_groups)) and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}, mean {numeric_field}:")
            print(grouped_df.head())
        else:
            print("No categorical grouping field found for grouping.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Visualize the distribution of the numeric field if available
if record_set_ids and 'numeric_field' in locals() and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30, edgecolor='black')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.grid(False)
    plt.show()

    # If a group_field was found earlier, show boxplot (if >1 group)
    if 'group_field' in locals() and group_field and df[group_field].nunique()>1:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field, by=group_field, grid=False, rot=45)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we loaded the FAIR² dataset for Mass Spectrometry Analysis of Extracellular Vesicles using the `mlcroissant` library.
* We explored its record sets and fields, extracted tabular data, and performed sample EDA and visualization on numeric columns such as protein molecular weight or abundance.
* Grouping and normalization were demonstrated using available fields.
* This workflow can be extended to deeper biological or statistical analyses as required using the Croissant `@id` references for complete traceability and interoperability.